# THz Dataset Explorer

This notebook provides interactive exploration of THz transmission datasets.

**Features:**
- Load and inspect datasets from the datalake
- Visualize time domain pulses (reference and sample)
- Plot transfer functions (real/imaginary and magnitude/phase)
- Display material parameters for each sample
- Random sample selection with reproducible indices

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import sys

# Add src to path - works from both notebook dir and project root
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    src_path = notebook_dir.parent / 'src'
else:
    src_path = notebook_dir / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from simulate import simulate_transmission_ml
from utils import list_available_datasets
from training_config import Config

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

## 1. List Available Datasets

In [14]:
# List all datasets in datalake
datasets = list_available_datasets()

print("Available datasets in datalake:")
print("=" * 70)

if not datasets:
    print("No datasets found. Generate one using:")
    print("  python src/generate_dataset.py --quick")
else:
    for i, ds in enumerate(datasets, 1):
        print(f"{i}. {ds}")
        
        # Try to load metadata
        metadata_path = Path('datalake/datasets') / ds / 'metadata.json'
        if metadata_path.exists():
            with open(metadata_path, 'r') as f:
                meta = json.load(f)
            print(f"   Description: {meta.get('description', 'N/A')}")
            print(f"   Total samples: {meta.get('total_samples', 'N/A'):,}")
            print(f"   Generated: {meta.get('generation_date', 'N/A')[:10]}")
        print()

Available datasets in datalake:
1. production_v1

2. quick_test



## 2. Load Dataset

Choose a dataset from the list above and specify the split (train/val/test).

In [15]:
# ========================================
# CONFIGURATION - Edit these values
# ========================================

DATASET_NAME = 'production_v1'  # Change this to your dataset name
SPLIT = 'train'  # 'train', 'val', or 'test'

# ========================================

# Load dataset
dataset_path = Path('../datalake/datasets') / DATASET_NAME
split_file = dataset_path / f'{SPLIT}.npz'

if not split_file.exists():
    raise FileNotFoundError(f"Dataset not found: {split_file}")

print(f"Loading: {DATASET_NAME} ({SPLIT} split)")
print("=" * 70)

data = np.load(split_file, allow_pickle=True)

# Extract data - handle both old and new formats
if 'parameters/n' in data.files:
    # Old format with slash-separated keys
    params_n = data['parameters/n']
    params_kappa = data['parameters/kappa']
    params_d = data['parameters/d']
elif 'parameters' in data.files:
    # New format with nested dictionary
    params_dict = data['parameters'].item()
    params_n = params_dict['n']
    params_kappa = params_dict['kappa']
    params_d = params_dict['d']
else:
    raise KeyError("Could not find parameters in dataset. Available keys: " + str(data.files))

frequencies = data['frequencies']

# Compute ml_input from T_complex (real/imag channels)
T_complex = data['T_complex']
ml_input = np.stack([T_complex.real, T_complex.imag], axis=1)  # [N, 2, M+1]

# Check if time domain data is available
has_time_domain = 'time_domain' in data.files
if has_time_domain:
    time_domain = data['time_domain']
    reference_pulse = data['reference_pulse']
    print("✓ Time domain data available")
else:
    print("⚠ Time domain data not saved in dataset")
    print("  Will regenerate from parameters using forward model")

# Load metadata
metadata_path = dataset_path / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    config_sim = metadata['simulation_config']
    L = config_sim['L']
    deltat = config_sim['deltat']
else:
    # Use defaults from Config
    config = Config()
    L = config.L
    deltat = config.DELTAT

n_samples = len(params_n)

print(f"\n✓ Loaded {n_samples:,} samples")
print(f"  Frequencies: {len(frequencies):,} points ({frequencies[1]/1e12:.2f} - {frequencies[-1]/1e12:.2f} THz)")
print(f"  ML input shape: {ml_input.shape}")
if has_time_domain:
    print(f"  Time domain shape: {time_domain.shape}")
print(f"\n  Simulation parameters:")
print(f"    L = {L} time samples")
print(f"    Δt = {deltat*1e12:.1f} fs")
print(f"    Time window = {L*deltat*1e12:.1f} ps")

Loading: production_v1 (train split)
⚠ Time domain data not saved in dataset
  Will regenerate from parameters using forward model

✓ Loaded 50,000 samples
  Frequencies: 8,193 points (0.00 - 5.00 THz)
  ML input shape: (50000, 2, 8193)

  Simulation parameters:
    L = 4096 time samples
    Δt = 0.1 fs
    Time window = 409.6 ps


## 3. Dataset Statistics

In [16]:
print("Parameter Statistics")
print("=" * 70)

print(f"\nRefractive Index (n):")
print(f"  Range: [{params_n.min():.3f}, {params_n.max():.3f}]")
print(f"  Mean:  {params_n.mean():.3f} ± {params_n.std():.3f}")
print(f"  Median: {np.median(params_n):.3f}")

print(f"\nExtinction Coefficient (κ):")
print(f"  Range: [{params_kappa.min():.4f}, {params_kappa.max():.4f}]")
print(f"  Mean:  {params_kappa.mean():.4f} ± {params_kappa.std():.4f}")
print(f"  Median: {np.median(params_kappa):.4f}")

d_um = params_d * 1e6
print(f"\nThickness (d):")
print(f"  Range: [{d_um.min():.1f}, {d_um.max():.1f}] μm")
print(f"  Mean:  {d_um.mean():.1f} ± {d_um.std():.1f} μm")
print(f"  Median: {np.median(d_um):.1f} μm")

Parameter Statistics

Refractive Index (n):
  Range: [1.100, 7.000]
  Mean:  4.052 ± 1.703
  Median: 4.050

Extinction Coefficient (κ):
  Range: [-0.1000, 0.0100]
  Mean:  -0.0448 ± 0.0317
  Median: -0.0446

Thickness (d):
  Range: [100.0, 1000.0] μm
  Mean:  549.7 ± 259.3 μm
  Median: 549.3 μm


## 4. Select Random Sample

Select a random sample from the dataset. The index is printed so you can reproduce the same sample later.

In [17]:
# Select random sample
# To reproduce a specific sample, set: sample_idx = <specific_index>
sample_idx = np.random.randint(0, n_samples)

# Extract sample data
n_sample = params_n[sample_idx]
kappa_sample = params_kappa[sample_idx]
d_sample = params_d[sample_idx]

# Get transmission coefficient (real/imag from ml_input)
T_real = ml_input[sample_idx, 0, :]
T_imag = ml_input[sample_idx, 1, :]
T_complex = T_real + 1j * T_imag

# Compute magnitude and phase
T_magnitude = np.abs(T_complex)
T_phase = np.angle(T_complex)  # Wrapped to [-π, π]

print("=" * 70)
print("SELECTED SAMPLE")
print("=" * 70)
print(f"\nDataset: {DATASET_NAME} ({SPLIT})")
print(f"Sample Index: {sample_idx} (out of {n_samples:,} samples)")
print(f"\nMaterial Parameters:")
print(f"  Refractive Index (n):      {n_sample:.4f}")
print(f"  Extinction Coeff (κ):      {kappa_sample:.6f}")
print(f"  Thickness (d):             {d_sample*1e6:.2f} μm ({d_sample*1e3:.4f} mm)")

# Calculate some derived quantities
c = 299792458  # Speed of light (m/s)
echo_delay = 2 * n_sample * d_sample / c  # Round trip time
print(f"\nDerived Quantities:")
print(f"  Expected echo delay:       {echo_delay*1e12:.2f} ps")
print(f"  Absorption (κ < 0):        {'Yes' if kappa_sample < 0 else 'No (gain)'}")
print(f"  Fresnel reflection coeff:  {((n_sample - 1) / (n_sample + 1)):.4f}")

print("\n" + "=" * 70)

SELECTED SAMPLE

Dataset: production_v1 (train)
Sample Index: 19193 (out of 50,000 samples)

Material Parameters:
  Refractive Index (n):      5.7590
  Extinction Coeff (κ):      0.000955
  Thickness (d):             315.51 μm (0.3155 mm)

Derived Quantities:
  Expected echo delay:       12.12 ps
  Absorption (κ < 0):        No (gain)
  Fresnel reflection coeff:  0.7041



In [24]:
# Interactive 3D visualization of parameter space coverage
import plotly.graph_objects as go

# Subsample for performance if dataset is large
max_points = 50000
if n_samples > max_points:
    idx = np.random.choice(n_samples, max_points, replace=False)
    plot_n = params_n[idx]
    plot_kappa = params_kappa[idx]
    plot_d = d_um[idx]
    title_suffix = f' (showing {max_points:,} of {n_samples:,} samples)'
else:
    plot_n = params_n
    plot_kappa = params_kappa
    plot_d = d_um
    title_suffix = f' ({n_samples:,} samples)'

# Create interactive 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=plot_n,
    y=plot_kappa,
    z=plot_d,
    mode='markers',
    marker=dict(
        size=2,
        color=plot_d,
        colorscale='Viridis',
        opacity=0.6,
        colorbar=dict(title='Thickness (μm)')
    ),
    hovertemplate='n: %{x:.3f}<br>κ: %{y:.4f}<br>d: %{z:.1f} μm<extra></extra>'
)])

fig.update_layout(
    title=f'Parameter Space Coverage - {DATASET_NAME}' + title_suffix,
    scene=dict(
        xaxis_title='Refractive Index (n)',
        yaxis_title='Extinction Coefficient (κ)',
        zaxis_title='Thickness (μm)',
    ),
    width=800,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

print(f"Parameter space bounds:")
print(f"  n:     [{params_n.min():.2f}, {params_n.max():.2f}]")
print(f"  κ:     [{params_kappa.min():.4f}, {params_kappa.max():.4f}]")
print(f"  d:     [{d_um.min():.1f}, {d_um.max():.1f}] μm")

Parameter space bounds:
  n:     [1.10, 7.00]
  κ:     [-0.1000, 0.0100]
  d:     [100.0, 1000.0] μm


## 5. Generate/Load Time Domain Data

If time domain data wasn't saved in the dataset, we'll regenerate it using the forward model.

In [19]:
if has_time_domain:
    # Use pre-computed time domain data
    sample_pulse = time_domain[sample_idx]
    ref_pulse = reference_pulse
    print("✓ Using pre-computed time domain data from dataset")
else:
    # Regenerate using forward model
    print("Regenerating time domain data using forward model...")
    
    freq, T_recomputed, ref_pulse, sample_pulse = simulate_transmission_ml(
        n=n_sample,
        kappa=kappa_sample,
        d=d_sample,
        L=L,
        deltat=deltat,
        device='cpu',
        return_time_domain=True,
        noise_level=None  # No noise for clean visualization
    )
    
    # Convert to numpy
    ref_pulse = ref_pulse.numpy()
    sample_pulse = sample_pulse.numpy()
    
    print("✓ Time domain data regenerated")

# Create time array
N = 4 * L  # Full time domain length
time_array = np.arange(N) * deltat
time_ps = time_array * 1e12  # Convert to picoseconds

# Find main pulse location
main_pulse_idx = np.argmax(np.abs(ref_pulse))
main_pulse_time = main_pulse_idx * deltat * 1e12

print(f"\nTime domain information:")
print(f"  Total time points: {N:,}")
print(f"  Time window: {time_ps[-1]:.2f} ps")
print(f"  Main pulse at: {main_pulse_time:.2f} ps (index {main_pulse_idx})")
print(f"  Expected 1st echo at: {main_pulse_time + echo_delay*1e12:.2f} ps")

Regenerating time domain data using forward model...
✓ Time domain data regenerated

Time domain information:
  Total time points: 16,384
  Time window: 1638.30 ps
  Main pulse at: 9.30 ps (index 93)
  Expected 1st echo at: 21.42 ps


## 6. Comprehensive Visualization

Create a multi-panel plot showing:
- Time domain: Reference and sample pulses
- Time domain: Echo region
- Transfer function: Magnitude and phase
- Transfer function: Real and imaginary parts

In [20]:
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)

freq_THz = frequencies / 1e12  # Convert to THz

# Color scheme
color_ref = '#2E86AB'
color_sample = '#A23B72'
color_mag = '#F18F01'
color_phase = '#C73E1D'

# =============================================================================
# Row 1: Time Domain - Main Pulse
# =============================================================================
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(time_ps[:L], ref_pulse, color=color_ref, linewidth=1.5, label='Reference pulse', alpha=0.8)
ax1.plot(time_ps[:L], sample_pulse[:L], color=color_sample, linewidth=1.5, label='Sample pulse', alpha=0.8)
ax1.set_xlabel('Time (ps)', fontsize=11)
ax1.set_ylabel('Amplitude (arb. units)', fontsize=11)
ax1.set_title(f'Time Domain - Main Pulse Region\n' + 
              f'Sample {sample_idx}: n={n_sample:.3f}, κ={kappa_sample:.4f}, d={d_sample*1e6:.1f}μm',
              fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([main_pulse_time - 5, main_pulse_time + 30])

# Add annotation for sample parameters
info_text = f'Dataset: {DATASET_NAME} ({SPLIT})\nIndex: {sample_idx}'
ax1.text(0.02, 0.98, info_text, transform=ax1.transAxes,
         fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

# =============================================================================
# Row 2, Left: Time Domain - Echo Region
# =============================================================================
ax2 = fig.add_subplot(gs[1, 0])

# Calculate expected echo positions
echo1_time = main_pulse_time + echo_delay * 1e12
echo2_time = main_pulse_time + 2 * echo_delay * 1e12
echo3_time = main_pulse_time + 3 * echo_delay * 1e12

ax2.plot(time_ps, sample_pulse, color=color_sample, linewidth=1, alpha=0.8)
ax2.axvline(x=main_pulse_time, color='gray', linestyle='-', alpha=0.4, linewidth=1.5, label='Main pulse')
ax2.axvline(x=echo1_time, color='red', linestyle='--', alpha=0.6, linewidth=1.5, label=f'1st echo ({echo1_time:.1f} ps)')
ax2.axvline(x=echo2_time, color='orange', linestyle='--', alpha=0.6, linewidth=1.5, label=f'2nd echo ({echo2_time:.1f} ps)')
ax2.axvline(x=echo3_time, color='green', linestyle='--', alpha=0.6, linewidth=1.5, label=f'3rd echo ({echo3_time:.1f} ps)')

ax2.set_xlabel('Time (ps)', fontsize=11)
ax2.set_ylabel('Amplitude', fontsize=11)
ax2.set_title('Time Domain - Echo Region', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([echo1_time - 10, min(echo3_time + 10, time_ps[-1])])

# =============================================================================
# Row 2, Right: Time Domain - Full Trace (Log Scale)
# =============================================================================
ax3 = fig.add_subplot(gs[1, 1])

ax3.semilogy(time_ps, np.abs(sample_pulse) + 1e-10, color=color_sample, linewidth=0.8, alpha=0.7)
ax3.axvline(x=main_pulse_time, color='gray', linestyle='-', alpha=0.4, linewidth=1)
ax3.axvline(x=echo1_time, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax3.axvline(x=echo2_time, color='orange', linestyle='--', alpha=0.4, linewidth=1)

ax3.set_xlabel('Time (ps)', fontsize=11)
ax3.set_ylabel('|Amplitude| (log scale)', fontsize=11)
ax3.set_title('Full Time Trace (Log Scale)', fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3, which='both')
ax3.set_xlim([0, min(100, time_ps[-1])])
ax3.set_ylim([1e-6, 1])

# =============================================================================
# Row 3, Left: Transfer Function - Magnitude and Phase
# =============================================================================
ax4 = fig.add_subplot(gs[2, 0])

ax4_twin = ax4.twinx()

# Magnitude
ln1 = ax4.plot(freq_THz, T_magnitude, color=color_mag, linewidth=2, label='|T(ω)| Magnitude', alpha=0.8)
ax4.set_xlabel('Frequency (THz)', fontsize=11)
ax4.set_ylabel('|T(ω)|', fontsize=11, color=color_mag)
ax4.tick_params(axis='y', labelcolor=color_mag)
ax4.set_xlim([0, 5])
ax4.grid(True, alpha=0.3)

# Phase
ln2 = ax4_twin.plot(freq_THz, T_phase, color=color_phase, linewidth=2, label='∠T(ω) Phase', alpha=0.8)
ax4_twin.set_ylabel('∠T(ω) (radians)', fontsize=11, color=color_phase)
ax4_twin.tick_params(axis='y', labelcolor=color_phase)
ax4_twin.set_ylim([-np.pi, np.pi])
ax4_twin.axhline(y=0, color='k', linestyle='--', alpha=0.2, linewidth=0.8)

ax4.set_title('Transfer Function - Magnitude and Phase', fontsize=11, fontweight='bold')

# Combined legend
lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax4.legend(lns, labs, loc='upper right', fontsize=9)

# =============================================================================
# Row 3, Right: Transfer Function - Real and Imaginary
# =============================================================================
ax5 = fig.add_subplot(gs[2, 1])

ax5.plot(freq_THz, T_real, color='blue', linewidth=2, label='Re(T)', alpha=0.8)
ax5.plot(freq_THz, T_imag, color='red', linewidth=2, label='Im(T)', alpha=0.8)
ax5.axhline(y=0, color='k', linestyle='--', alpha=0.3, linewidth=0.8)

ax5.set_xlabel('Frequency (THz)', fontsize=11)
ax5.set_ylabel('T(ω)', fontsize=11)
ax5.set_title('Transfer Function - Real and Imaginary Components', fontsize=11, fontweight='bold')
ax5.legend(loc='upper right', fontsize=9)
ax5.grid(True, alpha=0.3)
ax5.set_xlim([0, 5])

# Overall title
fig.suptitle(f'THz Dataset Explorer - Sample {sample_idx}', 
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("Plot generated successfully!")
print(f"To reproduce this exact sample, set: sample_idx = {sample_idx}")
print("=" * 70)

<IPython.core.display.Javascript object>


Plot generated successfully!
To reproduce this exact sample, set: sample_idx = 19193


/var/folders/q2/m5_vfjhs3m5b5szm6q3l9dlr0000gn/T/ipykernel_16063/1325491193.py:122: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



## 7. Generate New Random Sample

Run this cell to select a new random sample and re-run the visualization above.

In [21]:
# This will trigger a new random sample when you re-run cells 4-6
print("Run cells 4-6 again to visualize a new random sample")
print("Or manually set sample_idx in cell 4 to reproduce a specific sample")

Run cells 4-6 again to visualize a new random sample
Or manually set sample_idx in cell 4 to reproduce a specific sample


## 8. Compare Multiple Samples

Overlay multiple samples to compare their characteristics.

In [22]:
# Select multiple random samples
n_compare = 5
compare_indices = np.random.choice(n_samples, size=n_compare, replace=False)

print(f"Comparing {n_compare} samples:")
print(f"Indices: {compare_indices}")
print()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Comparison of {n_compare} Random Samples from {DATASET_NAME}', 
             fontsize=14, fontweight='bold')

colors = plt.cm.tab10(np.linspace(0, 1, n_compare))

for i, idx in enumerate(compare_indices):
    n_s = params_n[idx]
    kappa_s = params_kappa[idx]
    d_s = params_d[idx]
    
    T_r = ml_input[idx, 0, :]
    T_i = ml_input[idx, 1, :]
    T_c = T_r + 1j * T_i
    T_mag = np.abs(T_c)
    T_ph = np.angle(T_c)
    
    label = f'#{idx}: n={n_s:.2f}, κ={kappa_s:.3f}, d={d_s*1e6:.0f}μm'
    
    # Magnitude
    axes[0, 0].plot(freq_THz, T_mag, color=colors[i], linewidth=1.5, alpha=0.8, label=label)
    
    # Phase
    axes[0, 1].plot(freq_THz, T_ph, color=colors[i], linewidth=1.5, alpha=0.8, label=label)
    
    # Real
    axes[1, 0].plot(freq_THz, T_r, color=colors[i], linewidth=1.5, alpha=0.8, label=label)
    
    # Imaginary
    axes[1, 1].plot(freq_THz, T_i, color=colors[i], linewidth=1.5, alpha=0.8, label=label)
    
    print(f"  Sample {idx}: n={n_s:.3f}, κ={kappa_s:.4f}, d={d_s*1e6:.1f}μm")

# Configure subplots
axes[0, 0].set_xlabel('Frequency (THz)')
axes[0, 0].set_ylabel('|T(ω)|')
axes[0, 0].set_title('Magnitude')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim([0, 5])

axes[0, 1].set_xlabel('Frequency (THz)')
axes[0, 1].set_ylabel('∠T(ω) (rad)')
axes[0, 1].set_title('Phase')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim([0, 5])
axes[0, 1].set_ylim([-np.pi, np.pi])

axes[1, 0].set_xlabel('Frequency (THz)')
axes[1, 0].set_ylabel('Re(T)')
axes[1, 0].set_title('Real Part')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim([0, 5])
axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)

axes[1, 1].set_xlabel('Frequency (THz)')
axes[1, 1].set_ylabel('Im(T)')
axes[1, 1].set_title('Imaginary Part')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim([0, 5])
axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print(f"Comparison of samples: {list(compare_indices)}")
print("=" * 70)

Comparing 5 samples:
Indices: [16417  5704  3615 43674  8914]



<IPython.core.display.Javascript object>

/var/folders/q2/m5_vfjhs3m5b5szm6q3l9dlr0000gn/T/ipykernel_16063/2333461724.py:74: UserWarning:

Glyph 8736 (\N{ANGLE}) missing from font(s) Arial.



  Sample 16417: n=2.504, κ=-0.0114, d=317.7μm
  Sample 5704: n=5.009, κ=-0.0056, d=464.3μm
  Sample 3615: n=4.400, κ=-0.0528, d=203.4μm
  Sample 43674: n=2.534, κ=0.0018, d=371.6μm
  Sample 8914: n=5.480, κ=-0.0262, d=893.7μm

Comparison of samples: [16417, 5704, 3615, 43674, 8914]


## 9. Parameter Space Exploration

Visualize the distribution of parameters and their relationships.

In [23]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'Parameter Space - {DATASET_NAME} ({SPLIT})', fontsize=14, fontweight='bold')

# Histograms
axes[0, 0].hist(params_n, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
axes[0, 0].axvline(params_n.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {params_n.mean():.3f}')
axes[0, 0].set_xlabel('Refractive Index (n)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('n Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(params_kappa, bins=50, alpha=0.7, edgecolor='black', color='coral')
axes[0, 1].axvline(params_kappa.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {params_kappa.mean():.4f}')
axes[0, 1].set_xlabel('Extinction Coefficient (κ)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('κ Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].hist(d_um, bins=50, alpha=0.7, edgecolor='black', color='mediumseagreen')
axes[0, 2].axvline(d_um.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {d_um.mean():.1f} μm')
axes[0, 2].set_xlabel('Thickness (μm)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('d Distribution')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Scatter plots (correlations)
axes[1, 0].scatter(params_n, params_kappa, alpha=0.3, s=10)
axes[1, 0].set_xlabel('Refractive Index (n)')
axes[1, 0].set_ylabel('Extinction Coefficient (κ)')
axes[1, 0].set_title('n vs κ')
axes[1, 0].grid(True, alpha=0.3)
# Add correlation coefficient
corr_nk = np.corrcoef(params_n, params_kappa)[0, 1]
axes[1, 0].text(0.05, 0.95, f'Correlation: {corr_nk:.3f}', transform=axes[1, 0].transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

axes[1, 1].scatter(params_n, d_um, alpha=0.3, s=10)
axes[1, 1].set_xlabel('Refractive Index (n)')
axes[1, 1].set_ylabel('Thickness (μm)')
axes[1, 1].set_title('n vs d')
axes[1, 1].grid(True, alpha=0.3)
corr_nd = np.corrcoef(params_n, d_um)[0, 1]
axes[1, 1].text(0.05, 0.95, f'Correlation: {corr_nd:.3f}', transform=axes[1, 1].transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

axes[1, 2].scatter(params_kappa, d_um, alpha=0.3, s=10)
axes[1, 2].set_xlabel('Extinction Coefficient (κ)')
axes[1, 2].set_ylabel('Thickness (μm)')
axes[1, 2].set_title('κ vs d')
axes[1, 2].grid(True, alpha=0.3)
corr_kd = np.corrcoef(params_kappa, d_um)[0, 1]
axes[1, 2].text(0.05, 0.95, f'Correlation: {corr_kd:.3f}', transform=axes[1, 2].transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.tight_layout()
plt.show()

print("Parameter correlations:")
print(f"  n vs κ:  {corr_nk:.4f}")
print(f"  n vs d:  {corr_nd:.4f}")
print(f"  κ vs d:  {corr_kd:.4f}")
print("\n(Values near 0 indicate independence, as expected for uniform sampling)")

<IPython.core.display.Javascript object>

Parameter correlations:
  n vs κ:  0.0003
  n vs d:  0.0031
  κ vs d:  0.0067

(Values near 0 indicate independence, as expected for uniform sampling)


## Summary

This notebook provides interactive exploration of THz transmission datasets:

- **Cell 4-6**: Explore individual samples with full time/frequency domain visualization
- **Cell 8**: Compare multiple samples side-by-side
- **Cell 9**: Analyze parameter space distributions and correlations

**Tips:**
- To reproduce a specific sample, note its index and set `sample_idx = <index>` in Cell 4
- Re-run cells 4-6 to visualize different random samples
- Look for echoes in the time domain (vertical lines mark expected positions)
- The transfer function shows both magnitude/phase and real/imaginary representations

**Next Steps:**
- Use these visualizations to understand your data before training
- Check for any anomalies or unexpected patterns
- Compare different datasets to see the effect of parameter ranges
- Verify that noise levels and parameter distributions match your expectations